# Python in the Smoke 💨 🐍 💨
## Applied notebook: APIs, JSON, and London theatre at the V&A

---

**Repo:** [`python-in-the-smoke`](https://github.com/MarkGotham/python-in-the-smoke)

**By:** Mark Gotham

**Licence:** MIT

**Colour and symbol key:**
- 🟢 A block of information.
- 🟣 A core exercise.
- ⭐ An optional exercise.
- ✅ The solution to the exercise above it.
- 📝 A note with important information.
- ⚠️ Common errors, and how to fix them.
- `Grey text like this` shows an exact code or command. Type it exactly as shown, including any capital letters, spaces, and punctuation.

**Assumed knowledge:**

- This notebook builds on the foundational Part I notebooks.
- No other applied notebook is required.
- We use dictionaries, lists and f-strings (plus one loop in an optional exercise).
- We return to pandas selection, category counts and plotting.
- We use file paths to save and reload data.

**Learning Objectives**: After completing this notebook, you will be able to ...
- Make a simple API request with a search term
- Explain JSON and navigate the returned dictionaries and lists
- Build a small dataframe from museum catalogue records
- Investigate how different objects relate to London theatre
- Save a dated response and reuse it without another request
- Distinguish a search result from a representative sample

---
## 🟢 Information: Our cultural question

**What kinds of objects can tell us about London theatre?**

Think beyond scripts and stage buildings.
- A costume can document performance and design.
- A photograph can record a production or a place.
- A poster can show how theatre was promoted.
- A drawing can preserve something that no longer exists.

We will search the V&A catalogue for `London theatre`.
Then we will examine a small page of results.

Our question is exploratory.
We are asking what these records suggest, not taking a census of London theatre.

**Your first task:** name two kinds of object you expect to find.
Keep your prediction to compare with the results.

---
## 🟢 Information: Why use an API?

**API** stands for Application Programming Interface.
- It lets one piece of software interact with another.
- Here, Python asks the V&A for catalogue records.
- We can then explore those records with familiar pandas tools.

Institutions decide what their APIs expose and under what conditions.
Some require an account or charge for access.
This example uses a public search endpoint without credentials.

📝 APIs provide access but do not guarantee complete or error-free data.
We will inspect the records and their limits as we go.

---
## 🟢 Information: Make one search

We need the `requests` package.
- If you don't yet have it, run `%pip install requests` as below.
- If in doubt, run the cell below (no harm in checking).
- To review installation and import, see Notebook f.

For now, run the short example below.
We will unpack each part after seeing the result.

**First collection:** run this section once.
**Later sessions:** after saving your response, start at “Reload your saved response”.
You do not need to repeat the live request.

In [ ]:
%pip install requests

In [ ]:
import requests
import pandas as pd

In [ ]:
url = "https://api.vam.ac.uk/v2/objects/search"
params = {"q": "London theatre", "page_size": 20}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()
data = response.json()

In [ ]:
first = data["records"][0]
print("Title:", first["_primaryTitle"])
print("Object type:", first["objectType"])
print("Date:", first["_primaryDate"])

Great! We've already retrieved an object from a real museum catalogue.

Now we'll go through how this worked and ask some related questions:
- Was this the kind of object you predicted?
- What might this object tell us about theatre?
- What would you need to see or read before making that claim?

---
## 🟢 Information: What happened in the request?

Look again at the `requests.get` line in the code above:
- `url` is the address of the search (just like visiting an internet page from a browser)
- `params` is dictionary specifying our search, here comprising:
    - `q`: The search words
    - `page_size`: The number of records requested for this page
- `timeout=30`: Sometimes the request will fail. It's better to find out after a fixed period of time than to wait forever.

The `requests.get(...)` command sends a `GET` request (i.e., asks for information)
- the `response` is the server’s reply (with data or failing)
- `response.raise_for_status()` checks that the request worked. If the server replied with an error (for example 404 not found, or 500 server error), Python stops here with an `HTTPError` instead of carrying on with a failed response.
- Finally, `response.json()` decodes the data from "JSON" format (more on this below) into Python data.

📝 Notes:
- If no records are returned, `records[0]` cannot select a first item.
To check, try `len(data["records"])` before continuing.
- Requests also handles special characters in the URL for us safely.

---
### 🟣 Core Exercise: Inspect the response

Display the HTTP status code and the URL that Requests sent.
Then use `len(data["records"])` to count the records on this page.

---
---
---

### ✅ Solution: Inspect the response

In [ ]:
response.status_code

In [ ]:
response.url

In [ ]:
len(data["records"])

A successful request should normally have status 200.
The URL contains your search parameters in encoded form.
The returned page should have 20 records for this search, while that many matches remain available.

A page length is not the total number of matches.
We will inspect that distinction shortly.

---
## 🟢 Information: What is JSON?

**JSON** stands for "JavaScript Object Notation".
- It is a text format for structured data.
- APIs often use it to send records.
- Python decodes that text into values we can work with.

It often looks and behaves like a Python dictionary, but the two are not identical. Some differences:

| JSON            | Python after decoding |
|-----------------|-----------------------|
| object `{...}`  | `dict`                |
| array `[...]`   | `list`                |
| string          | `str`                 |
| number          | `int` or `float`      |
| `true`, `false` | `True`, `False`       |
| `null`          | `None`                |

JSON uses double quotes for strings and keys.
It does not allow comments or trailing commas.

This is for information only: we almost never need to type out JSON by hand;
`response.json()` has already decoded it for us.

The variable `data` we created with `data = response.json()` stored Python values, not the original JSON text.
This really is a dict:

In [ ]:
type(data)

In [ ]:
data.keys()

For this endpoint, the top-level value is a dictionary.
- `records` holds the returned object summaries.
- `info` describes the search results.
- `clusters` holds grouped information from the search.

We only need `records` and a little of `info` here.
Other APIs use other key names.
There is no universal rule that records must be called `records`.

---
## 🟢 Information: A dictionary containing a list

Follow the structure one step at a time.

1. `data` is a dictionary.
2. `data["records"]` is a list.
3. `data["records"][0]` is its first item.
4. That item is another dictionary: one object summary.

These are the same dictionary keys and list indices from Notebook b.
They are simply nested inside each other.

In [ ]:
records = data["records"]
type(records)

In [ ]:
first = records[0]
first.keys()

In [ ]:
first["_primaryTitle"]

The underscore at the start of `_primaryTitle` is part of the field name.
Type it exactly as supplied by the API.

These fields are **summaries**.
A primary title, maker or date can simplify a more detailed catalogue record.
An empty value can mean that this summary field has no text.
It does not prove the object has no history.

---
### 🟣 Core Exercise: Read another object

Select the second record.
Print its primary title, object type and primary date.
Then print the maker’s name.
The maker is nested: look inside `second["_primaryMaker"]` for the key `name`.
Some records have no maker, so use `.get(...)` to avoid a `KeyError` (the solution shows how).

---
---
---

### ✅ Solution: Read another object

In [ ]:
second = records[1]
print(second["_primaryTitle"])
print(second["objectType"])
print(second["_primaryDate"])

In [ ]:
maker = second.get("_primaryMaker", {})
print(maker.get("name", "No maker listed"))

The maker field is another nested dictionary.
The first `.get` selects it (or an empty dictionary if it is missing).
The second selects its `name` (or our fallback text).

`.get(key, default)` returns the default if the key is absent, whereas square brackets raise a `KeyError`.

If the maker name is blank or “Unknown”, report that.
Do not infer a maker from the title.
These are summary values, so open the full catalogue entry before interpreting them closely.

---
## 🟢 Information: See the cultural object

Numbers and labels are useful, but they are not the whole object.
The system number identifies a catalogue record.
Use it to construct a link to the V&A’s collection website.

Open the printed link in your browser.
- Read the description.
- Look at any available image.
- Note the maker, date and relationship to theatre.
- Check whether the record contains several objects or parts.

An image is not available for every object.
A descriptive record can still be useful evidence.

In [ ]:
system_number = first["systemNumber"]
object_url = f"https://collections.vam.ac.uk/item/{system_number}/"
print(object_url)

---
### 🟣 Core Exercise: A close reading

Choose one record from this page.
Print its title and collection link.
Open that entry and write two sentences in the Markdown cell below the code cell (double-click it to edit):
- What does the object contribute to a history of theatre?
- What remains uncertain from the record?

*Your two sentences here.*

---
---
---

### ✅ Solution: A close reading

In [ ]:
chosen = records[0]  # Change this index to choose another record.
print(chosen["_primaryTitle"])
print(f"https://collections.vam.ac.uk/item/{chosen['systemNumber']}/")

There is no single model interpretation.

A good answer names the object and uses evidence from its catalogue entry.
For example, a drawing of a theatre exterior may show how a building was represented.
It cannot by itself establish how an audience experienced a performance.

State whether your observation comes from the image, description or metadata.
These forms of evidence are related, but they are not interchangeable.

---
## 🟢 Information: From object summaries to a table

Now return to familiar pandas work.
A list of dictionaries can become a dataframe.
- One returned record becomes one row.
- Its keys become column names.
- Some values are still dictionaries or lists.

We will select a few simple columns first.
There is no need to flatten every part of the response.

In [ ]:
objects = pd.DataFrame(records)
columns = ["systemNumber", "_primaryTitle", "objectType", "_primaryDate", "_primaryPlace"]
objects[columns].head()

Read these as catalogue summaries.
- `_primaryDate` can include approximate dates or date ranges.
- Keep it as text for this exercise.
- `_primaryPlace` is not automatically the place depicted.
- A London match may come from another field in the full record.

The keyword search looks across catalogue fields.
“London theatre” does not mean every result’s title contains those words.
It also does not mean every object was made in London.

---
### 🟣 Core Exercise: Which object types appear?

Count the returned records for each `objectType`.
Then display the title, type and date of records in one category you find interesting.

---
---
---

### ✅ Solution: Which object types appear?

In [ ]:
type_counts = objects["objectType"].value_counts()
type_counts

In [ ]:
chosen_type = objects["objectType"].iloc[0]  # Replace with a type from your counts.
objects[objects["objectType"] == chosen_type][["_primaryTitle", "objectType", "_primaryDate"]]

Your counts describe this returned page.
They may change as the catalogue or search results change.

Compare the types with your opening prediction.
Which kinds of theatre history are visible here?
What kinds might be absent?
An absence on this page is not evidence that the museum holds no examples.

---
## 🟢 Information: A quick visual comparison

A bar chart makes the mix of object types easier to inspect.
We use the pandas plotting methods from Part I.

If matplotlib is missing, run `%pip install matplotlib` in a separate cell.
No new plotting package is required.

The title must make the scope clear: these are search results on one page.
A category count chart describes the catalogue records, not the number of performances.

In [ ]:
import matplotlib.pyplot as plt

ax = type_counts.sort_values().plot.barh(figsize=(8, 4), color="teal")
ax.set_title("Object types on one V&A search page: London theatre")
ax.set_xlabel("Number of returned records")
ax.set_ylabel("Object type")
ax.set_xlim(left=0)
ax.set_xticks(range(0, int(type_counts.max()) + 1))
plt.tight_layout()
plt.show()
plt.close(ax.get_figure())

---
### 🟣 Core Exercise: Interpret the pattern

Write three short sentences in the Markdown cell below (double-click it to edit):
1. Name the most frequent object type and its count on this page.
2. Suggest what that type could contribute to theatre history.
3. State a limit of using this page as evidence.

*Your three sentences here.*

---
---
---

### ✅ Solution: Interpret the pattern

Use the values from your own response.

Example structure:
- “[Type] appears in [n] of the [page length] returned records.”
- “These objects could help us investigate [a specific cultural question].”
- “This is one page of search results, not a representative sample of theatre history.”

The numerical observation should support a cultural question.
It should not replace looking at the objects themselves.

---
## 🟢 Information: A page is not the whole collection

The response includes information about the search.
- `record_count` is the reported total number of matches.
- `record_count_exact` says whether that total is exact or an estimate (if the response includes it).
- `page_size` is the size reported for the returned page (if the response includes it).
- `len(records)` is how many records we actually received.

We request a small page so we can learn with manageable data.
We do not need to download every match for this lesson.

Read the actual response rather than assuming that counts or page settings always agree.

In [ ]:
info = data["info"]
print("Reported matches:", info.get("record_count"))
print("Is that count exact?", info.get("record_count_exact"))
print("Records on this page:", len(records))
print("Reported page size:", info.get("page_size"))

**What our findings can support**
- Examples of objects returned by this search.
- Counts and comparisons within this page.
- Questions to investigate through catalogue descriptions.

**What they cannot establish**
- The most common object type in the whole museum.
- The complete history of London theatre.
- Whether theatre was more important in one place or period.

Search terms, cataloguing practices and result ordering shape what we see.
Several records can also relate to the same production or object series.
They are not necessarily independent examples of different cultural activity.

---
## 🟢 Information: Save once for later work

Save the decoded response as JSON.
This lets you return to the same records without making another request.

We need two tools that come with Python:
- `json` writes and reads JSON files.
- `date` supplies today’s date for the filename.

We will also record the endpoint and search parameters in a short source note.
There is no automatic load-or-download function here.
Collection and local analysis are separate, visible steps.

The following cell creates a `vam_snapshots` folder next to the notebook.
Check `Path.cwd()` if your working directory is elsewhere, as in Notebook g.

📝 **On Colab:** files saved in a Colab session are deleted when it ends. Download your JSON file and source note (Files panel in the left sidebar) before closing, or plan to repeat the live search next time.

In [ ]:
from pathlib import Path
from datetime import date
import json

Path("vam_snapshots").mkdir(exist_ok=True)
collection_date = date.today().isoformat()
slug = params["q"].lower().replace(" ", "-")
snapshot_path = Path(f"vam_snapshots/vam_{slug}_{collection_date}.json")
print(snapshot_path)

The filename combines a short version of the search term with the date, for example `vam_london-theatre_2026-11-10.json`.
It tells us what was searched and when this page was collected.
Two different searches on the same day get different filenames.

`json.dumps` turns Python values into JSON text (the `s` is for *string*).
- `ensure_ascii=False` keeps text characters readable.
- `indent=2` makes the saved structure easier to inspect.
- `encoding="utf-8"` (on the file) supports text characters.

We create the text *first* and open the file *second*.
If the conversion failed, no half-written file would be left behind.

Mode `"x"` creates a new file and refuses to overwrite one.
We also check `exists()` first, so re-running the cell is safe: an existing snapshot is kept and a message says so.
Nothing is silently overwritten, and delete a snapshot only deliberately.

If you change the search term, the filename changes with it, so the new response gets its own file.
The JSON and its note are checked separately, so if a run was interrupted, the next run fills in whatever is missing.

---
### 🟣 Core Exercise: Save the response

Save `data` to `snapshot_path`.
First check whether the file already exists: if it does, print a message and do not write.
Otherwise, convert `data` to text with `json.dumps` and write it.
Then do the same for a short source note beside it.
Include the endpoint, search parameters and collection date.
The note text is supplied in the first cell below (run it), so this step stays short.
You write the two files.

In [ ]:
source_note = (
    "Source: V&A Collections API\n"
    f"Endpoint: {url}\n"
    f"Parameters: {params}\n"
    f"Collected: {collection_date}\n"
    "Scope: one returned search page; not the complete collection.\n"
)
note_path = snapshot_path.with_suffix(".txt")
print(source_note)

---
---
---

### ✅ Solution: Save the response

In [ ]:
if snapshot_path.exists():
    print("Already saved, so not overwritten:", snapshot_path)
else:
    text = json.dumps(data, ensure_ascii=False, indent=2)
    with snapshot_path.open("x", encoding="utf-8") as file:
        file.write(text)
    print("Saved:", snapshot_path)

In [ ]:
if note_path.exists():
    print("Already saved, so not overwritten:", note_path)
else:
    with note_path.open("x", encoding="utf-8") as file:
        file.write(source_note)
    print("Saved:", note_path)

Keep the JSON file and its note together.
The date records collection time, not the date of the objects or their cataloguing.

For a later study, you may want a more precise retrieval time.
A date is sufficient for this introductory exercise.
Check the V&A’s reuse terms and the rights attached to individual images before publishing material.
Public API access is not itself an unrestricted image licence.

Attribution: these records come from the V&A Collections API (https://developers.vam.ac.uk/). Credit the V&A when you share your work.

---
## 🟢 Information: Reload your saved response

**For later sessions, start here.**
- Run the imports in the next cell.
- Replace the example filename with your own saved filename.
- Load the JSON file.
- Rebuild the dataframe using the familiar cells above.
- Continue exploring without calling `requests.get`.

`json.load` reads from an open file.
We wrote the same JSON text with `json.dumps`.
Neither needs internet access.

Opening a collection link in your browser still needs a connection.
The local analysis uses only the saved records.

No snapshot yet (for example, a fresh Colab session)?
The cell below falls back to a supplied sample file in `sample_data/`.
On Colab, download that file from the repo and upload it to the session first.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import requests  # only needed for the optional exercises below

# Replace this example with the filename printed when you saved your response.
# It contains your search term and the collection date.
saved_path = Path("vam_snapshots/vam_london-theatre_2026-11-10.json")

# No snapshot of your own? Use the sample file supplied with this notebook.
if not saved_path.exists():
    saved_path = Path("sample_data/vam_london_theatre_sample.json")
    print("Your snapshot was not found. Using the supplied sample:", saved_path)

with saved_path.open("r", encoding="utf-8") as file:
    saved_data = json.load(file)

records = saved_data["records"]
objects = pd.DataFrame(records)
objects[["_primaryTitle", "objectType", "_primaryDate"]].head()

---
### 🟣 Core Exercise: Check and continue locally

Count the object types again using the reloaded dataframe.
Compare with your earlier counts.
Then choose another object for a close reading.

---
---
---

### ✅ Solution: Check and continue locally

In [ ]:
objects["objectType"].value_counts()

The same saved records give the same counts.
This is why saving matters: you can change your analysis while holding the source data fixed.

Keep the returned records intact.
If you clean or reorganise the table, save that derived version under another name.
A CSV can be useful for sharing a table, but retain the JSON and source note too.

---
### ⭐ Optional Exercise: Try a different cultural search

Choose another search, such as `Covent Garden` or `London transport`.
Make one new request for 20 records.
Print a few titles and types.
Compare the cultural material it brings into view.
Keep this response in a new variable so you do not overwrite your first search.

---
---
---

### ✅ Solution: Try a different cultural search

In [ ]:
other_params = {"q": "Covent Garden", "page_size": 20}
other_response = requests.get("https://api.vam.ac.uk/v2/objects/search", params=other_params, timeout=30)
other_response.raise_for_status()
other_data = other_response.json()

for record in other_data["records"][:5]:
    print(record["_primaryTitle"], "—", record["objectType"])

A different search can foreground another part of the collection.
These are two selected result pages, not comparable population samples.

Discuss one useful discovery and one effect of your search wording.
If you keep working with this response, save it under a different filename with its own source note.

---
### ⭐ Optional Exercise: Flatten nested fields

Some dataframe cells still hold dictionaries, such as `_primaryMaker`.
Try `pd.json_normalize(records)` to flatten them into separate columns.
Then display the title and `_primaryMaker.name` for the first few rows, and count the most frequent makers.
Check how missing makers are handled.

---
---
---

### ✅ Solution: Flatten nested fields

In [ ]:
flat = pd.json_normalize(records)
flat[["_primaryTitle", "_primaryMaker.name"]].head()

In [ ]:
flat["_primaryMaker.name"].value_counts(dropna=False).head()

Nested keys become column names joined by a dot.
By default `value_counts` leaves out missing values; `dropna=False` shows them.
A blank or missing maker is a finding about the catalogue summary, not an error.
A dominant maker may reflect a series of related objects, not independent examples.

---
### ⭐ Optional Exercise: Request a second page

The `page` parameter selects which page of results to return.
Make one new request with `"page": 2` and the same search and page size.
Print five titles, and check whether each `systemNumber` also appeared on page 1.
Use a new variable so you do not overwrite your first response.

---
---
---

### ✅ Solution: Request a second page

In [ ]:
page_two_params = {"q": "London theatre", "page_size": 20, "page": 2}
page_two_response = requests.get("https://api.vam.ac.uk/v2/objects/search", params=page_two_params, timeout=30)
page_two_response.raise_for_status()
page_two_records = page_two_response.json()["records"]

page_one_numbers = [record["systemNumber"] for record in records]
for record in page_two_records[:5]:
    print(record["_primaryTitle"], "| also on page 1:", record["systemNumber"] in page_one_numbers)

Pages normally continue where the last one stopped, so overlap should be rare.
Result order can change between requests as the catalogue is updated.
More pages mean more records, not a representative sample.

---
## ⚠️ Common Errors

| Symptom | What to check |
|---|---|
| `ModuleNotFoundError` for requests | Run `%pip install requests` in this notebook’s environment |
| `ConnectionError` or `Timeout` | Check your internet connection and endpoint; return to a saved response if you have one |
| `HTTPError` | Read the status and message; the service may be unavailable or refusing the request |
| `JSONDecodeError` | The response body is not valid JSON; inspect `response.status_code` and a short excerpt of `response.text` |
| `KeyError` | Check exact keys, including underscores, and inspect `data.keys()` or `first.keys()` |
| `IndexError` at `[0]` | Check that the returned records list is not empty |
| `FileExistsError` while saving | The file exists and mode `"x"` refused to overwrite it; our save cells check `exists()` first, so check that test is still there |
| `FileNotFoundError` while reloading | Check the saved filename and working directory with `Path.cwd()` |

A decoded JSON response can still contain an error message.
Check the response structure before treating it as catalogue data.
Do not repeatedly retry an unchanged failing request.

A blank title is a data issue, not a Python error.
Use the object type and catalogue link to investigate it.

**The cells below fail on purpose.**
Read each message, work out the cause, then compare with the explanation underneath.

### ⚠️ Deliberate error 1: a mistyped key

In [ ]:
first["primaryTitle"]

`KeyError: 'primaryTitle'`: the key is `_primaryTitle`, with the leading underscore.
Fix it by copying the exact name from `first.keys()`.

### ⚠️ Deliberate error 2: selecting from an empty list

In [ ]:
no_results = []
no_results[0]

`IndexError: list index out of range`: an empty list has no first item.
This is what `data["records"][0]` does if a search matches nothing.
Check `len(...)` before selecting.

### ⚠️ Deliberate error 3: overwriting a saved file

In [ ]:
# Mode "x" refuses to overwrite. This uses the supplied sample file, so your own snapshot is safe.
with Path("sample_data/vam_london_theatre_sample.json").open("x", encoding="utf-8") as file:
    file.write("this would overwrite the file")

`FileExistsError`: the file is already there, and mode `"x"` protects it.
That is the behaviour we want for evidence.
To make a new file, choose a new filename.
This is why the save cells check `exists()` first: re-running them is safe, and the error does not happen by accident.

---
## 🟢 Information: Summary

| To ... | Write ... |
|---|---|
| Search an API | `requests.get(url, params=params, timeout=30)` |
| Check HTTP errors | `response.raise_for_status()` |
| Decode JSON | `data = response.json()` |
| Select the returned page | `records = data["records"]` |
| Select an object | `records[0]` |
| Build a dataframe | `pd.DataFrame(records)` |
| Count object types | `objects["objectType"].value_counts()` |
| Save JSON | `text = json.dumps(data, ensure_ascii=False, indent=2)`, then write `text` to a file |
| Avoid overwriting | `if path.exists():` print a message, `else:` write with mode `"x"` |
| Read saved JSON | `json.load(file)` |

You have used an API to find cultural material, not just to practise requests.
Keep your close reading beside your numerical findings.

**Final reflection:**
- What did these objects make you notice about London theatre?
- What would you search for next?
- What evidence would you need before making a broader claim?

### Source and documentation

- [V&A collections search](https://collections.vam.ac.uk/)
- [V&A API quick start](https://developers.vam.ac.uk/guide/v2/quick-start.html)
- [V&A search guide](https://developers.vam.ac.uk/guide/v2/search/introduction.html)
- [Requests quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/)
- [Python JSON documentation](https://docs.python.org/3/library/json.html)

The live search endpoint and summary fields were checked while preparing this draft on 14 October 2026.
Results and documentation can change.
There are deliberately no fixed counts or catalogue interpretations in the solutions.

End

---